# 20260923 Visualize Core

Use this notebook for full-session checks: behavior in the arena, arena/startbox ROI overlays, head direction, and optional neural maps for curated cells. Keep trial-by-trial comparisons in `20260923_visualize_trials.ipynb`.

## Imports

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocess_functions.manifest import load_session_records
from preprocess_functions.pipeline import default_aligned_session_path, default_cue_events_path, default_trials_path
from preprocess_functions import plot, roi

## Config

In [ ]:
OUTPUT_ROOT = Path("preprocess_out")
MANIFEST = OUTPUT_ROOT / "manifest_with_cells.csv"
if not MANIFEST.exists():
    MANIFEST = OUTPUT_ROOT / "manifest_with_h5.csv"
if not MANIFEST.exists():
    MANIFEST = Path("data_paths/RSC_PPC_Cohort1_paths.xlsx")

SHEET = None
LAB_DRIVE = os.environ.get("LAB_DRIVE_PATH") or None
RECORDING_ID = None
FPS = 30.0

ROI_NAMES = ["arena", "startbox_L", "startbox_R"]
LOAD_OR_COLLECT_ROIS = True
SAVE_ROI_FEATURES_TO_ALIGNED_CSV = False

TRAJECTORY_DOWNSAMPLE = 5
HD_DOWNSAMPLE = 20
CELL_COL = None

## Load Recording

In [ ]:
records = load_session_records(MANIFEST, sheet_name=SHEET, lab_drive=LAB_DRIVE)
records_df = pd.DataFrame([
    {
        "recording_id": record.recording_id,
        "session_id": record.session_id,
        "mouse_id": record.mouse_id,
        "trial_type": record.trial_type,
        "aligned_csv": str(default_aligned_session_path(OUTPUT_ROOT, record)),
        "cell_csv": str(record.cell_csv) if record.cell_csv else None,
    }
    for record in records
])
display(records_df)

In [ ]:
def choose_record(records, recording_id=None):
    if recording_id is None:
        return records[0]
    for record in records:
        if recording_id in {record.recording_id, record.session_id, record.trial_type, record.mouse_id}:
            return record
    raise ValueError(f"No manifest row matched {recording_id!r}.")


record = choose_record(records, RECORDING_ID)
aligned_csv = default_aligned_session_path(OUTPUT_ROOT, record)
if not aligned_csv.exists():
    raise FileNotFoundError(f"Missing aligned CSV. Run build_aligned_sessions first: {aligned_csv}")

df = pd.read_csv(aligned_csv)
print("recording_id:", record.recording_id)
print("aligned CSV:", aligned_csv)
print("shape:", df.shape)
display(df.head())

## Arena ROIs

In [ ]:
def choose_roi_id(record, roi_names, root=PROJECT_ROOT):
    for candidate in [record.recording_id, record.session_id]:
        if candidate and any(roi.roi_json_path(candidate, name, root=root).exists() for name in roi_names):
            return candidate
    return record.recording_id


def behavior_video_path(record, df):
    if "beh_vid_path" in df.columns and df["beh_vid_path"].notna().any():
        values = df["beh_vid_path"].dropna().astype(str).str.strip()
        values = values[~values.str.lower().isin(["", "nan", "none", "null"])]
        if not values.empty:
            return Path(values.iloc[0])
    if record.beh_vid is not None:
        return Path(record.beh_vid)
    raise FileNotFoundError("No behavior video path found for ROI selection.")


masks = {}
arena_mask = None
roi_paths = {}

if LOAD_OR_COLLECT_ROIS:
    ROI_ID = choose_roi_id(record, ROI_NAMES)
    video_path = behavior_video_path(record, df)
    print("behavior video:", video_path)

    rois, roi_paths = roi.load_or_collect_named_rois(
        video_path=video_path,
        session_id=ROI_ID,
        roi_names=ROI_NAMES,
        root=PROJECT_ROOT,
        folder_name="arena_rois",
    )
    height, width = roi.get_video_hw(video_path)
    masks = roi.build_roi_masks(rois, height, width)
    arena_mask = masks.get("arena")

    if {"ear_mid_x", "ear_mid_y"}.issubset(df.columns):
        df = roi.add_roi_features(df, rois, ref_x="ear_mid_x", ref_y="ear_mid_y")
        df = roi.add_arena_only_column(df)
        if SAVE_ROI_FEATURES_TO_ALIGNED_CSV:
            aligned_csv.parent.mkdir(parents=True, exist_ok=True)
            df.to_csv(aligned_csv, index=False)
            print("updated aligned CSV:", aligned_csv)

    print("ROI JSONs:")
    for name, path in roi_paths.items():
        print(f"  {name}: {path}")
else:
    print("ROI loading disabled.")

## Core Behavior Plots

In [ ]:
plot.plot_trajectory(
    df,
    masks=masks or None,
    x_col="ear_mid_x",
    y_col="ear_mid_y",
    downsample=TRAJECTORY_DOWNSAMPLE,
)

In [ ]:
if "head_dir_rad" in df.columns:
    plot.plot_hd_trajectory(
        df,
        masks=masks or None,
        x_col="ear_mid_x",
        y_col="ear_mid_y",
        angle_col="head_dir_rad",
        downsample=HD_DOWNSAMPLE,
    )

## Optional Neural Overview

In [ ]:
cell_cols = [col for col in df.columns if col.startswith("registered_cell_")]
if not cell_cols:
    cell_cols = [col for col in df.columns if col.startswith("cell_")]
if CELL_COL is None and cell_cols:
    CELL_COL = cell_cols[0]
print("available cells:", len(cell_cols))
print("selected cell:", CELL_COL)

In [ ]:
if CELL_COL is not None and arena_mask is not None:
    plot.plot_2d_ratemap(
        df,
        cell_col=CELL_COL,
        arena_mask=arena_mask,
        bins=20,
    )

In [ ]:
if CELL_COL is not None and "head_dir_rad" in df.columns:
    plot.plot_hd(df, cell_col=CELL_COL, head_dir_col="head_dir_rad", n_bins=36)

In [ ]:
required_for_ebc = {"ear_mid_x", "ear_mid_y", "nose.x", "nose.y", "head_dir_rad"}
if CELL_COL is not None and arena_mask is not None and required_for_ebc.issubset(df.columns):
    plot.plot_cell_summary(
        df,
        cell_col=CELL_COL,
        arena_mask=arena_mask,
        egocentric_smooth_sigma=(1.0, 1.5),
        head_direction_bins=36,
    )